In [ ]:
#Workflow

# 1. Imports
# 2. User settings 
# 3. Input validation and reference loading
# 4. Alignment and read orientation
# 5. FASTA parsing and unique-read filtering
# 6. Nucleotide mutation calling
# 7. Optional amino-acid annotation
# 8. Dataset-level mutation calling and summary
# 9. Run the analysis
# 10. Preview output tables.
# 11. Save CSV outputs

# Install dependencies with:
# pip install "biopython" "pandas"


In [ ]:
## 1. Imports

from __future__ import annotations

from collections import defaultdict
from pathlib import Path
from typing import Optional, Union

import pandas as pd
from Bio import Align, SeqIO
from Bio.Seq import Seq

PathLike = Union[str, Path]


In [ ]:
## 2. User settings
#
# Edit only the values in this cell when analyzing a new dataset.
#
# READS_FASTA:
#     FASTA file containing sequencing reads.
#
# REFERENCE_FILE:
#     Reference sequence in FASTA or GenBank format.
#
# OUTPUT_DIR:
#     Directory in which output CSV files will be written.
#
# OUTPUT_PREFIX:
#     Prefix added to each output filename.
#
# MIN_COUNT:
#     Minimum occurrence count required for a unique read.
#
# LENGTH_TOLERANCE:
#     Allowed fractional difference between a read and the reference length.
#     For example, 0.10 retains reads within +/-10% of the reference length.
#
# ORIENTATION_WINDOW:
#     Number of bases used to determine read orientation.
#
# CDS_START and CDS_END:
#     1-based coding-region boundaries used for amino-acid annotation.
#     If the full reference is coding sequence, leave CDS_START = 1 and
#     CDS_END = None.

READS_FASTA = Path("reads.fasta")
REFERENCE_FILE = Path("reference.fasta")

OUTPUT_DIR = Path("output")
OUTPUT_PREFIX = "sample"

MIN_COUNT = 15
LENGTH_TOLERANCE = 0.10
ORIENTATION_WINDOW = 50

CDS_START = 1
CDS_END: Optional[int] = None


In [ ]:
## 3. Input validation and reference loading

def validate_settings(
    reads_path: PathLike,
    reference_path: PathLike,
    min_count: int,
    length_tolerance: float,
    orientation_window: int,
) -> None:
    """Validate user-supplied file paths and basic analysis parameters."""
    reads_path = Path(reads_path)
    reference_path = Path(reference_path)

    if not reads_path.is_file():
        raise FileNotFoundError(f"Reads FASTA not found: {reads_path}")

    if not reference_path.is_file():
        raise FileNotFoundError(f"Reference file not found: {reference_path}")

    if min_count < 1:
        raise ValueError("MIN_COUNT must be at least 1.")

    if not 0 <= length_tolerance < 1:
        raise ValueError("LENGTH_TOLERANCE must be between 0 and 1.")

    if orientation_window < 1:
        raise ValueError("ORIENTATION_WINDOW must be at least 1.")


def load_reference(path: PathLike) -> str:
    """Load a single reference sequence from FASTA or GenBank."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in {".gb", ".gbk", ".genbank"}:
        file_format = "genbank"
    elif suffix in {".fa", ".fasta", ".fna"}:
        file_format = "fasta"
    else:
        raise ValueError(
            "Reference file must be FASTA (.fa, .fasta, .fna) "
            "or GenBank (.gb, .gbk, .genbank)."
        )

    record = SeqIO.read(path, file_format)
    reference = str(record.seq).upper()

    if not reference:
        raise ValueError("Reference sequence is empty.")

    return reference


In [ ]:
## 4. Alignment and read orientation

def make_aligner() -> Align.PairwiseAligner:
    """Create the pairwise aligner used for orientation and mutation calling."""
    aligner = Align.PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 2
    aligner.mismatch_score = -3
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -2
    aligner.end_gap_score = 0
    return aligner


def orient_read(
    read: str,
    reference: str,
    window: int = 50,
) -> str:
    """Orient a read relative to the reference sequence.

    The beginning of the read is compared in the forward and reverse-complement
    orientations. The orientation with the higher alignment score is retained.
    """
    read = read.upper()
    window_size = min(window, len(read), len(reference))

    if window_size == 0:
        return read

    aligner = make_aligner()

    forward_score = aligner.score(
        reference[:window_size],
        read[:window_size],
    )

    reverse_test = str(
        Seq(read[:window_size]).reverse_complement()
    )
    reverse_score = aligner.score(
        reference[-window_size:],
        reverse_test,
    )

    if forward_score >= reverse_score:
        return read

    return str(Seq(read).reverse_complement())


In [ ]:
## 5. FASTA parsing and unique-read filtering

def parse_fasta(
    fasta_path: PathLike,
    reference: str,
    orientation_window: int = 50,
) -> pd.DataFrame:
    """Read FASTA sequences, orient them, and collapse identical reads.

    Returns
    -------
    pandas.DataFrame
        Columns:
        - seq: oriented nucleotide sequence
        - occurrences: number of identical reads
        - length: sequence length in nucleotides
    """
    sequence_counts: dict[str, int] = defaultdict(int)

    for record in SeqIO.parse(str(fasta_path), "fasta"):
        sequence = orient_read(
            str(record.seq),
            reference,
            orientation_window,
        )
        sequence_counts[sequence] += 1

    if not sequence_counts:
        return pd.DataFrame(
            columns=["seq", "occurrences", "length"]
        )

    unique_reads = pd.DataFrame(
        [
            {
                "seq": sequence,
                "occurrences": count,
                "length": len(sequence),
            }
            for sequence, count in sequence_counts.items()
        ]
    )

    return (
        unique_reads
        .sort_values("occurrences", ascending=False)
        .reset_index(drop=True)
    )


def filter_unique_reads(
    unique_reads: pd.DataFrame,
    reference_length: int,
    length_tolerance: float = 0.10,
    min_count: int = 15,
) -> pd.DataFrame:
    """Filter unique reads by expected length and minimum occurrence count."""
    minimum_length = int(
        reference_length * (1 - length_tolerance)
    )
    maximum_length = int(
        reference_length * (1 + length_tolerance)
    )

    keep = (
        unique_reads["length"].between(
            minimum_length,
            maximum_length,
        )
        & (unique_reads["occurrences"] >= min_count)
    )

    return (
        unique_reads.loc[keep]
        .copy()
        .reset_index(drop=True)
    )


In [ ]:
## 6. Nucleotide mutation calling

def call_mutations(
    read: str,
    reference: str,
) -> list[dict[str, object]]:
    """Align one read to the reference and call nucleotide-level mutations.

    Mutation positions use 1-based reference coordinates.

    Returns
    -------
    list of dict
        Each dictionary describes one substitution, insertion, or deletion.

    Notes
    -----
    Insertions are reported after the indicated reference position.
    """
    aligner = make_aligner()
    alignment = aligner.align(reference, read)[0]
    coordinates = alignment.coordinates

    mutations: list[dict[str, object]] = []

    for index in range(coordinates.shape[1] - 1):
        ref_start = int(coordinates[0, index])
        ref_end = int(coordinates[0, index + 1])

        read_start = int(coordinates[1, index])
        read_end = int(coordinates[1, index + 1])

        ref_step = ref_end - ref_start
        read_step = read_end - read_start

        # Aligned sequence block: inspect individual bases for substitutions.
        if ref_step > 0 and read_step > 0:
            if ref_step != read_step:
                raise ValueError(
                    "Unexpected alignment block with unequal lengths."
                )

            for offset in range(ref_step):
                ref_base = reference[ref_start + offset]
                alt_base = read[read_start + offset]

                if ref_base == alt_base:
                    continue

                position = ref_start + offset + 1

                mutations.append(
                    {
                        "mutation_type": "substitution",
                        "position": position,
                        "ref": ref_base,
                        "alt": alt_base,
                        "nt_change": (
                            f"{ref_base}{position}{alt_base}"
                        ),
                    }
                )

        # Deletion in the read relative to the reference.
        elif ref_step > 0 and read_step == 0:
            deleted_sequence = reference[ref_start:ref_end]
            position = ref_start + 1

            mutations.append(
                {
                    "mutation_type": "deletion",
                    "position": position,
                    "ref": deleted_sequence,
                    "alt": "-",
                    "nt_change": (
                        f"del{position}-{ref_end}:"
                        f"{deleted_sequence}"
                    ),
                }
            )

        # Insertion in the read relative to the reference.
        elif ref_step == 0 and read_step > 0:
            inserted_sequence = read[read_start:read_end]
            position = ref_start

            mutations.append(
                {
                    "mutation_type": "insertion",
                    "position": position,
                    "ref": "-",
                    "alt": inserted_sequence,
                    "nt_change": (
                        f"ins_after_{position}:"
                        f"{inserted_sequence}"
                    ),
                }
            )

    return mutations


In [ ]:
## 7. Optional amino-acid annotation

def annotate_snv_aa(
    reference: str,
    position: int,
    ref_base: str,
    alt_base: str,
    cds_start: int = 1,
    cds_end: Optional[int] = None,
) -> Optional[str]:
    """Convert a single-nucleotide substitution to an amino-acid change.

    Insertions and deletions are not translated in this simplified workflow
    because their protein-level consequences depend on reading frame and event
    length.

    Parameters
    ----------
    reference:
        Full nucleotide reference sequence.
    position:
        1-based nucleotide position of the substitution.
    ref_base:
        Reference nucleotide.
    alt_base:
        Alternate nucleotide.
    cds_start:
        1-based first nucleotide of the coding sequence.
    cds_end:
        1-based final nucleotide of the coding sequence. If None, the reference
        sequence end is used.

    Returns
    -------
    str or None
        Amino-acid change such as "I66T" or synonymous notation such as "G36=".
    """
    if cds_end is None:
        cds_end = len(reference)

    if position < cds_start or position > cds_end:
        return None

    if len(ref_base) != 1 or len(alt_base) != 1:
        return None

    offset = position - cds_start
    codon_offset = (offset // 3) * 3

    coding_sequence = reference[cds_start - 1:cds_end]
    reference_codon = coding_sequence[
        codon_offset:codon_offset + 3
    ]

    if len(reference_codon) != 3:
        return None

    base_in_codon = offset % 3
    mutant_codon = list(reference_codon)
    mutant_codon[base_in_codon] = alt_base
    mutant_codon = "".join(mutant_codon)

    reference_aa = str(Seq(reference_codon).translate())
    alternate_aa = str(Seq(mutant_codon).translate())
    aa_position = (codon_offset // 3) + 1

    if reference_aa == alternate_aa:
        return f"{reference_aa}{aa_position}="

    return f"{reference_aa}{aa_position}{alternate_aa}"


In [ ]:
## 8. Dataset-level mutation calling and summary

def call_mutations_for_unique_reads(
    unique_reads: pd.DataFrame,
    reference: str,
    cds_start: int = 1,
    cds_end: Optional[int] = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Call mutations for every retained unique read.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame]
        mutation_calls:
            One row per mutation per unique sequence.
        mutant_sequences:
            One row per unique sequence with its complete mutation set.
    """
    mutation_rows: list[dict[str, object]] = []
    mutant_rows: list[dict[str, object]] = []

    for row in unique_reads.itertuples(index=False):
        sequence = row.seq
        count = int(row.occurrences)

        mutations = call_mutations(
            sequence,
            reference,
        )

        mutation_labels: list[str] = []

        for mutation in mutations:
            aa_change = None

            if mutation["mutation_type"] == "substitution":
                aa_change = annotate_snv_aa(
                    reference=reference,
                    position=int(mutation["position"]),
                    ref_base=str(mutation["ref"]),
                    alt_base=str(mutation["alt"]),
                    cds_start=cds_start,
                    cds_end=cds_end,
                )

            mutation_rows.append(
                {
                    "occurrences": count,
                    **mutation,
                    "aa_change": aa_change,
                    "read_seq": sequence,
                }
            )

            mutation_labels.append(
                str(mutation["nt_change"])
            )

        mutant_rows.append(
            {
                "seq": sequence,
                "occurrences": count,
                "mutation_count": len(mutations),
                "mutations": (
                    "; ".join(mutation_labels)
                    if mutations
                    else "WT"
                ),
            }
        )

    mutation_calls = pd.DataFrame(
        mutation_rows,
        columns=[
            "occurrences",
            "mutation_type",
            "position",
            "ref",
            "alt",
            "nt_change",
            "aa_change",
            "read_seq",
        ],
    )

    mutant_sequences = pd.DataFrame(
        mutant_rows,
        columns=[
            "seq",
            "occurrences",
            "mutation_count",
            "mutations",
        ],
    )

    return mutation_calls, mutant_sequences


def summarize_mutations(
    mutation_calls: pd.DataFrame,
) -> pd.DataFrame:
    """Sum read occurrences for each distinct mutation."""
    output_columns = [
        "mutation_type",
        "position",
        "ref",
        "alt",
        "nt_change",
        "aa_change",
        "occurrences",
    ]

    if mutation_calls.empty:
        return pd.DataFrame(columns=output_columns)

    mutation_summary = (
        mutation_calls
        .groupby(
            [
                "mutation_type",
                "position",
                "ref",
                "alt",
                "nt_change",
                "aa_change",
            ],
            dropna=False,
        )["occurrences"]
        .sum()
        .reset_index()
        .sort_values(
            ["occurrences", "position"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    return mutation_summary


In [ ]:
## 9. Run the analysis

validate_settings(
    reads_path=READS_FASTA,
    reference_path=REFERENCE_FILE,
    min_count=MIN_COUNT,
    length_tolerance=LENGTH_TOLERANCE,
    orientation_window=ORIENTATION_WINDOW,
)

reference = load_reference(REFERENCE_FILE)

print(f"Reference length: {len(reference):,} nt")

unique_reads_all = parse_fasta(
    fasta_path=READS_FASTA,
    reference=reference,
    orientation_window=ORIENTATION_WINDOW,
)

print(
    "Unique sequences before filtering: "
    f"{len(unique_reads_all):,}"
)

unique_reads = filter_unique_reads(
    unique_reads=unique_reads_all,
    reference_length=len(reference),
    length_tolerance=LENGTH_TOLERANCE,
    min_count=MIN_COUNT,
)

print(
    "Unique sequences after filtering: "
    f"{len(unique_reads):,}"
)

mutation_calls, mutant_sequences = (
    call_mutations_for_unique_reads(
        unique_reads=unique_reads,
        reference=reference,
        cds_start=CDS_START,
        cds_end=CDS_END,
    )
)

mutation_summary = summarize_mutations(
    mutation_calls
)

print(
    "Distinct mutation calls: "
    f"{len(mutation_summary):,}"
)


In [ ]:
## 10. Preview output tables

display(unique_reads.head())
display(mutant_sequences.head())
display(mutation_summary.head(20))


In [ ]:
## 11. Save CSV outputs

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

output_files = {
    "unique_reads": OUTPUT_DIR / (
        f"{OUTPUT_PREFIX}_unique_reads.csv"
    ),
    "mutant_sequences": OUTPUT_DIR / (
        f"{OUTPUT_PREFIX}_mutant_sequences.csv"
    ),
    "mutation_calls": OUTPUT_DIR / (
        f"{OUTPUT_PREFIX}_mutation_calls.csv"
    ),
    "mutation_summary": OUTPUT_DIR / (
        f"{OUTPUT_PREFIX}_mutation_summary.csv"
    ),
}

unique_reads.to_csv(
    output_files["unique_reads"],
    index=False,
)
mutant_sequences.to_csv(
    output_files["mutant_sequences"],
    index=False,
)
mutation_calls.to_csv(
    output_files["mutation_calls"],
    index=False,
)
mutation_summary.to_csv(
    output_files["mutation_summary"],
    index=False,
)

print("Saved output files:")
for output_file in output_files.values():
    print(f"  {output_file}")


In [ ]:
## Output definitions
#
# *_unique_reads.csv
#     Unique oriented sequences retained after filtering, with occurrence
#     counts and sequence lengths.
#
# *_mutant_sequences.csv
#     One row per retained unique sequence, with the complete set of
#     nucleotide mutations carried by that sequence.
#
# *_mutation_calls.csv
#     One row per called mutation per unique sequence.
#
# *_mutation_summary.csv
#     Total occurrence count for each distinct mutation.
#
# Mutation notation:
#
# A100G
#     Reference A at nucleotide 100 changed to G.
#
# del100-102:ATG
#     Reference bases 100 through 102 were deleted.
#
# ins_after_100:ATG
#     ATG was inserted after reference nucleotide 100.
#
# All nucleotide positions use 1-based reference coordinates.
#
# Public-repository notes:
# - Input data are not bundled into this notebook.
# - No local user paths, credentials, or private identifiers are included.
# - The notebook contains no plotting or figure-generation functions.
# - A repository should also include a README, requirements file, and an
#   appropriate LICENSE selected by the authors/institution.
